In [168]:
from google import genai
from typing import TypedDict, Literal
from dotenv import load_dotenv
from langgraph.graph import StateGraph, START, END

In [169]:
load_dotenv()

client = genai.Client()

In [170]:
class QuadraticState(TypedDict):

    a: int
    b: int
    c: int

    equation: str
    discriminant: float
    result: str

In [171]:
def equation_creater(State: QuadraticState) -> QuadraticState:

    a = State['a']
    b = State['b']
    c = State['c']

    equation = f"{a}x**2 + {b}x + {c}"

    return {
        'equation': equation
    }

In [172]:
def discriminant_calculator(State: QuadraticState) -> QuadraticState:

    a = State['a']
    b = State['b']
    c = State['c']

    discriminant = (b**2) - (4*a*c)

    return {
        'discriminant': discriminant
    }

In [173]:
def real_root_calculator(State: QuadraticState) -> QuadraticState:

    a = State['a']
    b = State['b']
    c = State['c']
    d = State['discriminant']

    real_root_1 = ((-b) + (d**0.5)) / (2*a)
    real_root_2 = ((-b) - (d**0.5)) / (2*a)

    result = f"Two real roots are: {real_root_1} and {real_root_2}"

    return {
        'result': result
    }

In [174]:
def equal_root_calculator(State: QuadraticState) -> QuadraticState:

    a = State['a']
    b = State['b']
    c = State['c']

    real_root = (-b) / (2*a)

    result = f"Repeated root is {real_root}"

    return {
        'result': result
    }

In [175]:
def no_real_root_calculator(State: QuadraticState) -> QuadraticState:

    a = State['a']
    b = State['b']
    c = State['c']

    result = f"No real roots exists for these values: a: {a}, b: {b}, c: {c}"

    return {
        'result': result
    }

In [176]:
def check_condition(State: QuadraticState) -> Literal['real_root_calculator', 'equal_root_calculator', 'no_real_root_calculator']:

    if State['discriminant'] > 0 :
        return 'real_root_calculator'
    elif State['discriminant'] == 0 :
        return 'equal_root_calculator'
    else :
        return 'no_real_root_calculator'

In [177]:
graph = StateGraph(QuadraticState)

graph.add_node('equation_creater', equation_creater)
graph.add_node('discriminant_calculator', discriminant_calculator)
graph.add_node('real_root_calculator', real_root_calculator)
graph.add_node('equal_root_calculator', equal_root_calculator)
graph.add_node('no_real_root_calculator', no_real_root_calculator)

graph.add_edge(START, 'equation_creater')
graph.add_edge('equation_creater', 'discriminant_calculator')
graph.add_conditional_edges('discriminant_calculator', check_condition)
graph.add_edge('real_root_calculator', END)
graph.add_edge('equal_root_calculator', END)
graph.add_edge('no_real_root_calculator', END)

workflow = graph.compile()

In [179]:
initial_state = {
    'a': 2,
    'b': 4,
    'c': 1
}

final_state = workflow.invoke(initial_state)

final_state

{'a': 2,
 'b': 4,
 'c': 1,
 'equation': '2x**2 + 4x + 1',
 'discriminant': 8,
 'result': 'Two real roots are: -0.2928932188134524 and -1.7071067811865475'}